# Trial Wrangling Selector

Use this notebook to load trial data across animals, session types, cohorts, genotypes, and trial outcomes. Edit the clearly marked selection cells, then run downward to inspect the selected trial table and summary counts.

## 1. Setup

In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "analysis" / "daily_merge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from analysis.datasets import load_dataset_selections

BASE_DATA_DIR = ROOT / "DataFiles"
ROOT

PosixPath('/Users/mafaldavalente/Documents/Mafalda_analysis')

## 2. Choose Datasets

Start with one dataset for fast exploration, or set `DATASET_SELECTIONS = "all"` to load every discovered line/cohort folder. You can also provide an explicit list such as `[('CNTNAP2', 'cohort2'), ('SHANK3', 'cohort1')]`.

In [2]:
def discover_dataset_selections(base_dir=BASE_DATA_DIR):
    selections = []
    for path in sorted(Path(base_dir).glob("*_cohort*")):
        if not path.is_dir():
            continue
        line, _, cohort = path.name.partition("_cohort")
        if line and cohort:
            selections.append((line, f"cohort{cohort}"))
    return selections

AVAILABLE_DATASETS = discover_dataset_selections()
AVAILABLE_DATASETS

[('CNTNAP2', 'cohort1'),
 ('CNTNAP2', 'cohort2'),
 ('CNTNAP2', 'cohort3'),
 ('CNTNAP2', 'cohort4'),
 ('SHANK3', 'cohort1'),
 ('SHANK3', 'cohort2'),
 ('Stakes', 'cohort2')]

In [4]:
# =========================
# DATASET SELECTION
# =========================

# Use a list of (line, cohort) tuples, or set to "all" when you want every discovered dataset.
DATASET_SELECTIONS = [("Stakes", "cohort2")]

# Examples:
# DATASET_SELECTIONS = "all"
# DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]

SELECTED_DATASETS = AVAILABLE_DATASETS if DATASET_SELECTIONS == "all" else DATASET_SELECTIONS
SELECTED_DATASETS

[('Stakes', 'cohort2')]

## 3. Load Data And Show Available Values

In [5]:
df_all, meta_all, dataset_info = load_dataset_selections(
    selections=SELECTED_DATASETS,
    base_dir=str(BASE_DATA_DIR),
    require_meta=False,
)

for col in ["animal", "line", "cohort", "genotype", "dataset_key", "abort_type"]:
    if col in df_all.columns:
        df_all[col] = df_all[col].astype("string").str.strip()

def clean_abort_type(series):
    return series.astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

if "abort_type" in df_all.columns:
    df_all["abort_type"] = clean_abort_type(df_all["abort_type"])

if "success" in df_all.columns:
    success_num = pd.to_numeric(df_all["success"], errors="coerce")
    df_all["trial_outcome"] = np.select(
        [success_num.eq(1), success_num.eq(-1), success_num.eq(0)],
        ["success", "incorrect", "aborted"],
        default="unknown",
    )
else:
    df_all["trial_outcome"] = "unknown"

available = {
    "datasets": sorted(df_all.get("dataset_key", pd.Series(dtype="string")).dropna().unique().tolist()),
    "lines": sorted(df_all.get("line", pd.Series(dtype="string")).dropna().unique().tolist()),
    "cohorts": sorted(df_all.get("cohort", pd.Series(dtype="string")).dropna().unique().tolist()),
    "genotypes": sorted(df_all.get("genotype", pd.Series(dtype="string")).dropna().unique().tolist()),
    "animals": sorted(df_all.get("animal", pd.Series(dtype="string")).dropna().unique().tolist()),
    "session_types": sorted(pd.to_numeric(df_all.get("session_type", pd.Series(dtype=float)), errors="coerce").dropna().unique().tolist()),
    "training_levels": sorted(pd.to_numeric(df_all.get("training_level", pd.Series(dtype=float)), errors="coerce").dropna().unique().tolist()),
    "ramps": sorted(pd.to_numeric(df_all.get("sound_ramp_time", pd.Series(dtype=float)), errors="coerce").dropna().unique().tolist()),
    "trial_outcomes": sorted(df_all["trial_outcome"].dropna().unique().tolist()),
    "abort_types": sorted(df_all.get("abort_type", pd.Series(dtype="string")).dropna().unique().tolist()),
}

display(pd.DataFrame([(key, values) for key, values in available.items()], columns=["field", "available values"]))
print(f"Loaded {len(df_all):,} trials from {df_all['animal'].nunique():,} animals.")
dataset_info

,field,available values
0,datasets,[Stakes:cohort2]
1,lines,[Stakes]
2,cohorts,[cohort2]
3,genotypes,[wt]
4,animals,"[JCS0013, JCS0014, JCS0015, JCS0016, JCS0017, ..."
5,session_types,[1]
6,training_levels,"[1, 2, 3, 4, 5, 6]"
7,ramps,"[0.005, 0.02, 0.1, 0.2, 0.5]"
8,trial_outcomes,"[aborted, incorrect, success]"
9,abort_types,"[CNP, Fixation, LNP, MT+, MT-, RT+, RT-]"


Loaded 35,144 trials from 10 animals.


{'selections': [{'line': 'Stakes',
   'cohort': 'cohort2',
   'dataset_key': 'Stakes:cohort2'}],
 'dataset_keys': ['Stakes:cohort2'],
 'used_csvs': {'Stakes:cohort2': '/Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/Stakes_cohort2/merged_all_subjects.csv'},
 'missing_meta_dataset_keys': []}

## 4. Choose Trial Filters

Use `"all"` to keep everything for a field. Otherwise use lists. For trial type, use `TRIAL_OUTCOMES = ['success']`, `['incorrect']`, `['aborted']`, or any combination. `ABORT_TYPES` only narrows trials whose outcome is `aborted`.

In [8]:
# =========================
# TRIAL SELECTIONS
# =========================

ANIMALS = "all"          # column: animal; "all" or e.g. ["JCS0013", "JCS0014"]
LINES = "all"            # column: line; "all" or e.g. ["Stakes"]
COHORTS = "all"          # column: cohort; "all" or e.g. ["cohort2", "cohort3"]
GENOTYPES = "all"        # column: genotype; "all" or e.g. ["wt", "het", "hom"]
SESSION_TYPES = "all"    # column: session_type; "all" or e.g. [1, 2]
TRAINING_LEVELS = "all"  # column: training_level; "all", one number e.g. 16, list e.g. [7, 16], or range e.g. (7, 16)
RAMPS = [0.1]            # column: sound_ramp_time; "all" or e.g. [0.005, 0.01]

# Trial outcomes: derived column trial_outcome, created from column success.
# Values: "success", "incorrect", "aborted", "unknown".
TRIAL_OUTCOMES = ["success", "incorrect"]   # column: trial_outcome; "all" or e.g. ["success"] or ["success", "incorrect"]

# Abort types: use with aborted trials. Common values include "CNP", "Fixation", "LNP", "MT+", "RT-".
ABORT_TYPES = "all"      # column: abort_type; "all" or e.g. ["Fixation", "CNP"]

# Optional session ids if you want exact sessions rather than session types.
SESSIONS = "all"         # column: session; "all" or e.g. [1, 2, 3]

# Numeric range filters. Use (None, None) to keep all values.
# Bounds are inclusive and use the same units as the raw data columns.
TIMED_FIXATION_RANGE = (None, None)      # column: fix_time; e.g. (0.2, 1.5)
REACTION_TIME_RANGE = (None, None)       # column: timed_rt; e.g. (0.05, 3.0)
INTENDED_FIXATION_RANGE = (None, None)   # column: intended_fix_time; e.g. (0.1, 2.0)

selection = {
    "animals": ANIMALS,
    "lines": LINES,
    "cohorts": COHORTS,
    "genotypes": GENOTYPES,
    "session_types": SESSION_TYPES,
    "training_levels": TRAINING_LEVELS,
    "ramps": RAMPS,
    "trial_outcomes": TRIAL_OUTCOMES,
    "abort_types": ABORT_TYPES,
    "sessions": SESSIONS,
    "timed_fixation_range": TIMED_FIXATION_RANGE,
    "reaction_time_range": REACTION_TIME_RANGE,
    "intended_fixation_range": INTENDED_FIXATION_RANGE,
}
selection

{'animals': 'all',
 'lines': 'all',
 'cohorts': 'all',
 'genotypes': 'all',
 'session_types': 'all',
 'training_levels': 'all',
 'ramps': [0.1],
 'trial_outcomes': ['success', 'incorrect'],
 'abort_types': 'all',
 'sessions': 'all',
 'timed_fixation_range': (None, None),
 'reaction_time_range': (None, None),
 'intended_fixation_range': (None, None)}

## 5. Apply Filters

In [9]:
def as_list(values):
    if values == "all" or values is None:
        return None
    if isinstance(values, (str, int, float)):
        return [values]
    return list(values)

def apply_string_filter(mask, df, column, values):
    values = as_list(values)
    if values is None or column not in df.columns:
        return mask
    wanted = {str(value).strip() for value in values}
    return mask & df[column].astype("string").str.strip().isin(wanted)

def apply_numeric_filter(mask, df, column, values):
    values = as_list(values)
    if values is None or column not in df.columns:
        return mask
    wanted = pd.to_numeric(pd.Series(values), errors="coerce").dropna().tolist()
    observed = pd.to_numeric(df[column], errors="coerce")
    return mask & observed.isin(wanted)

def apply_numeric_selector_filter(mask, df, column, selector):
    if selector in (None, "all") or column not in df.columns:
        return mask
    observed = pd.to_numeric(df[column], errors="coerce")
    if isinstance(selector, tuple):
        return apply_numeric_range_filter(mask, df, column, selector)
    values = as_list(selector)
    wanted = pd.to_numeric(pd.Series(values), errors="coerce").dropna().tolist()
    return mask & observed.isin(wanted)

def apply_numeric_range_filter(mask, df, column, bounds):
    if bounds in (None, "all") or column not in df.columns:
        return mask
    if len(bounds) != 2:
        raise ValueError(f"{column} range must be a (min, max) tuple, e.g. (0.2, 1.5).")
    lower, upper = bounds
    if lower is None and upper is None:
        return mask
    observed = pd.to_numeric(df[column], errors="coerce")
    range_mask = observed.notna()
    if lower is not None:
        range_mask &= observed >= lower
    if upper is not None:
        range_mask &= observed <= upper
    return mask & range_mask

mask = pd.Series(True, index=df_all.index)
mask = apply_string_filter(mask, df_all, "animal", ANIMALS)
mask = apply_string_filter(mask, df_all, "line", LINES)
mask = apply_string_filter(mask, df_all, "cohort", COHORTS)
mask = apply_string_filter(mask, df_all, "genotype", GENOTYPES)
mask = apply_numeric_filter(mask, df_all, "session_type", SESSION_TYPES)
mask = apply_numeric_selector_filter(mask, df_all, "training_level", TRAINING_LEVELS)
mask = apply_numeric_filter(mask, df_all, "sound_ramp_time", RAMPS)
mask = apply_numeric_filter(mask, df_all, "session", SESSIONS)
mask = apply_string_filter(mask, df_all, "trial_outcome", TRIAL_OUTCOMES)
mask = apply_string_filter(mask, df_all, "abort_type", ABORT_TYPES)
mask = apply_numeric_range_filter(mask, df_all, "fix_time", TIMED_FIXATION_RANGE)
mask = apply_numeric_range_filter(mask, df_all, "timed_rt", REACTION_TIME_RANGE)
mask = apply_numeric_range_filter(mask, df_all, "intended_fix_time", INTENDED_FIXATION_RANGE)

selected_trials = df_all.loc[mask].copy()

print(f"Selected {len(selected_trials):,} / {len(df_all):,} trials ({len(selected_trials) / max(len(df_all), 1):.1%}).")
display(selected_trials.head(10))

Selected 3,134 / 35,144 trials (8.9%).


,animal,batch,experimenter,version,bias,repeated_trial,trial,trial_start,tared_trial_start,trial_end,...,rt_start_frame,mt_start_frame,lnp_start_frame,lnp_end_frame,cohort,line,sex,genotype,dataset_key,trial_outcome
7769,JCS0014,stakes,MV,0.11.1,-0.04,False,1918,3.866959e+09,76.645984,3.866959e+09,...,8552.0,8582.0,8604.0,8610.0,cohort2,Stakes,female,wt,Stakes:cohort2,success
7792,JCS0014,stakes,MV,0.11.1,0.20,False,1941,3.866959e+09,295.692992,3.866959e+09,...,29209.0,29224.0,29272.0,29704.0,cohort2,Stakes,female,wt,Stakes:cohort2,success
7804,JCS0014,stakes,MV,0.11.1,0.04,False,1953,3.866959e+09,380.829984,3.866959e+09,...,37667.0,37672.0,37698.0,37727.0,cohort2,Stakes,female,wt,Stakes:cohort2,incorrect
7812,JCS0014,stakes,MV,0.11.1,0.20,False,1961,3.866959e+09,451.940000,3.866959e+09,...,44496.0,44513.0,44535.0,44964.0,cohort2,Stakes,female,wt,Stakes:cohort2,success
7813,JCS0014,stakes,MV,0.11.1,0.24,False,1962,3.866959e+09,456.884992,3.866959e+09,...,45168.0,45187.0,45221.0,45580.0,cohort2,Stakes,female,wt,Stakes:cohort2,success
7829,JCS0014,stakes,MV,0.11.1,0.24,False,1978,3.866960e+09,598.940992,3.866960e+09,...,58814.0,58833.0,58872.0,59260.0,cohort2,Stakes,female,wt,Stakes:cohort2,success
7830,JCS0014,stakes,MV,0.11.1,0.28,False,1979,3.866960e+09,603.903008,3.866960e+09,...,59307.0,59312.0,59346.0,59772.0,cohort2,Stakes,female,wt,Stakes:cohort2,success
7833,JCS0014,stakes,MV,0.11.1,0.28,False,1982,3.866960e+09,623.599008,3.866960e+09,...,61340.0,61352.0,61391.0,61733.0,cohort2,Stakes,female,wt,Stakes:cohort2,success
7834,JCS0014,stakes,MV,0.11.1,0.24,False,1983,3.866960e+09,629.792992,3.866960e+09,...,61784.0,61786.0,61815.0,62039.0,cohort2,Stakes,female,wt,Stakes:cohort2,incorrect
7836,JCS0014,stakes,MV,0.11.1,0.28,False,1985,3.866960e+09,650.256992,3.866960e+09,...,63979.0,63980.0,64010.0,64391.0,cohort2,Stakes,female,wt,Stakes:cohort2,success


## 6. Data Structure Of Selected Trials

In [10]:
structure = pd.DataFrame({
    "column": selected_trials.columns,
    "dtype": [str(dtype) for dtype in selected_trials.dtypes],
    "non_null": selected_trials.notna().sum().to_numpy(),
    "missing": selected_trials.isna().sum().to_numpy(),
    "missing_%": (selected_trials.isna().mean().to_numpy() * 100).round(1),
    "n_unique": [selected_trials[col].nunique(dropna=True) for col in selected_trials.columns],
})

print(f"Selected trial table shape: {selected_trials.shape[0]:,} rows x {selected_trials.shape[1]:,} columns")
display(structure)

key_columns = [
    "dataset_key", "line", "cohort", "genotype", "animal", "session", "session_type",
    "trial", "trial_outcome", "success", "abort_type", "training_level", "fix_time",
    "timed_rt", "intended_fix_time", "ABL", "ILD", "sound_index", "stim_dur",
    "stim_dur_label", "response_poke", "repeated_trial",
]
existing_key_columns = [col for col in key_columns if col in selected_trials.columns]
display(selected_trials[existing_key_columns].head(20))

Selected trial table shape: 3,134 rows x 98 columns


,column,dtype,non_null,missing,missing_%,n_unique
0,animal,string,3134,0,0.0,9
1,batch,str,3134,0,0.0,1
2,experimenter,str,3134,0,0.0,1
3,version,str,3134,0,0.0,1
4,bias,float64,3134,0,0.0,31
...,...,...,...,...,...,...
93,line,string,3134,0,0.0,1
94,sex,str,3134,0,0.0,1
95,genotype,string,3134,0,0.0,1
96,dataset_key,string,3134,0,0.0,1


,dataset_key,line,cohort,genotype,animal,session,session_type,trial,trial_outcome,success,...,fix_time,timed_rt,intended_fix_time,ABL,ILD,sound_index,stim_dur,stim_dur_label,response_poke,repeated_trial
7769,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1918,success,1,...,264.224052,0.301216,260.303651,15.0,-30.0,15,6000,RT,-1.0,False
7792,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1941,success,1,...,232.543945,0.141984,229.462301,20.0,40.0,14,6000,RT,1.0,False
7804,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1953,incorrect,-1,...,240.479946,0.049664,236.519608,10.0,20.0,14,6000,RT,-1.0,False
7812,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1961,success,1,...,225.919724,0.164256,222.954903,10.0,-20.0,14,6000,RT,-1.0,False
7813,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1962,success,1,...,279.168129,0.196160,273.580430,10.0,20.0,16,6000,RT,1.0,False
7829,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1978,success,1,...,352.320194,0.187840,348.157539,10.0,20.0,13,6000,RT,1.0,False
7830,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1979,success,1,...,225.120068,0.056256,221.944793,10.0,20.0,15,6000,RT,1.0,False
7833,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1982,success,1,...,252.416134,0.123936,246.546803,20.0,40.0,15,6000,RT,1.0,False
7834,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1983,incorrect,-1,...,300.000191,0.019488,295.023583,20.0,40.0,13,6000,RT,-1.0,False
7836,Stakes:cohort2,Stakes,cohort2,wt,JCS0014,6,1,1985,success,1,...,465.280056,0.010848,462.058930,15.0,30.0,12,6000,RT,1.0,False


## 7. Summary Counts

In [9]:
def count_table(df, group_cols, label):
    group_cols = [col for col in group_cols if col in df.columns]
    if not group_cols:
        return pd.DataFrame({"summary": [label], "trials": [len(df)]})
    out = (
        df.groupby(group_cols, dropna=False)
          .size()
          .reset_index(name="trials")
          .sort_values("trials", ascending=False)
          .reset_index(drop=True)
    )
    return out

summary = pd.DataFrame({
    "metric": [
        "trials", "animals", "sessions", "session_types", "ramps", "datasets", "lines", "cohorts", "genotypes",
    ],
    "value": [
        len(selected_trials),
        selected_trials["animal"].nunique() if "animal" in selected_trials else pd.NA,
        selected_trials["session"].nunique() if "session" in selected_trials else pd.NA,
        selected_trials["session_type"].nunique() if "session_type" in selected_trials else pd.NA,
        pd.to_numeric(selected_trials["sound_ramp_time"], errors="coerce").dropna().nunique() if "sound_ramp_time" in selected_trials else pd.NA,
        selected_trials["dataset_key"].nunique() if "dataset_key" in selected_trials else pd.NA,
        selected_trials["line"].nunique() if "line" in selected_trials else pd.NA,
        selected_trials["cohort"].nunique() if "cohort" in selected_trials else pd.NA,
        selected_trials["genotype"].nunique() if "genotype" in selected_trials else pd.NA,
    ],
})

display(summary)

print("Trials by outcome")
display(count_table(selected_trials, ["trial_outcome", "abort_type"], "outcome"))

print("Trials by animal / genotype / session type")
display(count_table(selected_trials, ["dataset_key", "genotype", "animal", "session_type"], "animal_session_type"))

print("Trials by session")
display(count_table(selected_trials, ["dataset_key", "animal", "session", "session_type", "sound_ramp_time", "trial_outcome"], "session").head(200))

,metric,value
0,trials,0
1,animals,0
2,sessions,0
3,session_types,0
4,datasets,0
5,lines,0
6,cohorts,0
7,genotypes,0


Trials by outcome


,trial_outcome,abort_type,trials


Trials by animal / genotype / session type


,dataset_key,genotype,animal,session_type,trials


Trials by session


,dataset_key,animal,session,session_type,trial_outcome,trials


## 8. Optional: Save Selected Trials

In [ ]:
SAVE_SELECTED = False
OUTPUT_DIR = ROOT / "outputs" / "wrangled_trials"
OUTPUT_NAME = "selected_trials.csv"

if SAVE_SELECTED:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / OUTPUT_NAME
    selected_trials.to_csv(output_path, index=False)
    print(f"Saved {len(selected_trials):,} selected trials to {output_path}")
else:
    print("SAVE_SELECTED is False; no file written.")